# **5. Baseline Models (Logistic Regression, Decision Tree, Random Forest)**
Inputs: Day 4 artifacts `X_train`/`X_val`/`X_test` + `y_*` + `val_demographics_lookup`.
Baselines are evaluated on **X_val** (X_test reserved for Day 6 final + Day 9 fairness).

### **Overview & key decisions**
This notebook builds the baseline models on the notebook4 artifacts.
- **Evaluation partition:** all three baselines are scored on `X_val` (661,199 rows). `X_test` (1,652,997) is held out for the final Day-6 model and Day-9 fairness — it is never touched here.
- **Imbalance:** `class_weight='balanced'` (LR / DT) and `'balanced_subsample'` (RF), per the README "class weighting, not oversampling" decision.
- **Reproducibility:** every split uses `random_state=42` and is stratified on `approved`.
- **Artifacts written:** `day5_cluster_profile.csv`, `day5_vif_continuous.csv`, `day5_feature_selection.csv`, `day5_baseline_metrics.csv`, ROC/PR overlays, and `markdown/day5_baseline_summary.md`.


In [1]:
import numpy as np, pandas as pd, os
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve,
                             precision_recall_curve, silhouette_score)
from scipy.stats import spearmanr
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

M = '/Volumes/Mitul/Projects/home-mortage-approval-predictor/data/processed/modelling/'
FIG = '/Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/day5/'
os.makedirs(FIG, exist_ok=True)

X_train = pd.read_parquet(M + 'X_train.parquet').astype(np.float32)
X_val   = pd.read_parquet(M + 'X_val.parquet').astype(np.float32)
X_test  = pd.read_parquet(M + 'X_test.parquet').astype(np.float32)
y_train = pd.read_parquet(M + 'y_train.parquet')['approved'].astype('int8').values
y_val   = pd.read_parquet(M + 'y_val.parquet')['approved'].astype('int8').values
y_test  = pd.read_parquet(M + 'y_test.parquet')['approved'].astype('int8').values
val_demo = pd.read_parquet(M + 'val_demographics_lookup.parquet')

CONT_FEATURES = ['loan_amount','loan_to_value_ratio','property_value','income','debt_to_income_ratio',
 'loan_term','total_units','prepayment_penalty_term','intro_rate_period','tract_population',
 'tract_minority_population_percent','ffiec_msa_md_median_family_income','tract_to_msa_income_percentage',
 'tract_owner_occupied_units','tract_one_to_four_family_homes','tract_median_age_of_housing_units',
 'loan_to_income_ratio','debt_to_income_ratio_missing','prepayment_penalty_term_missing','intro_rate_period_missing']
print('loaded', X_train.shape, X_val.shape, X_test.shape)


loaded (5950786, 103) (661199, 103) (1652997, 103)


## **Step 1 - Baseline prep: schema parity + shared evaluator**
- **Schema parity:** confirm `X_train`, `X_val`, `X_test` share identical columns, order, and dtypes (103 float32 features) - the leakage-safe encode-on-train-only discipline from Day 4 must hold across all partitions.
- **Demographics alignment:** confirm `val_demographics_lookup` row-count matches `X_val` (protected attributes are kept out of the model matrix and used only for later fairness work).
- **`evaluate_model`:** one shared function returns ROC-AUC, PR-AUC, precision/recall/F1 at the 0.5 threshold, and the confusion matrix, so LR / DT / RF are compared on identical terms.


In [2]:
def schema_parity(*dfs):
    cols = [list(df.columns) for df in dfs]
    ok = all(c == cols[0] for c in cols)
    dt = all(list(df.dtypes) == list(dfs[0].dtypes) for df in dfs)
    print('columns identical:', ok, '| dtypes identical:', dt, '| n_features:', dfs[0].shape[1])
    return ok and dt

assert schema_parity(X_train, X_val, X_test)
assert len(val_demo) == len(X_val), 'val demo misalignment'
print('val demographics row-aligned to X_val: OK')


columns identical: True | dtypes identical: True | n_features: 103
val demographics row-aligned to X_val: OK


In [3]:
def evaluate_model(model, X, y, name='model'):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    r = dict(name=name,
             roc_auc=roc_auc_score(y, proba),
             pr_auc=average_precision_score(y, proba),
             precision=precision_score(y, pred),
             recall=recall_score(y, pred),
             f1=f1_score(y, pred),
             cm=confusion_matrix(y, pred),
             proba=proba)
    print(f"[{name}] ROC-AUC={r['roc_auc']:.4f} PR-AUC={r['pr_auc']:.4f} P={r['precision']:.4f} R={r['recall']:.4f} F1={r['f1']:.4f}")
    return r

results = {}


## **Step 0 - Unsupervised segmentation (clustering)**
Goal: find natural applicant/loan segments *before* using the label - exploratory context only.
- **Sample:** 1,000,000 rows drawn from `X_train`, stratified on `approved` (approval rate verified ~= 74.6%).
- **Preprocessing:** StandardScaler on the 20 continuous features; one-hot block left as-is; then **PCA to 26 components (90% variance)** so the ~83 one-hot dimensions don't dominate Euclidean distance by count.
- **k selection:** inertia + silhouette on a 15k subsample (50k would OOM on the n^2 distance matrix); **k = 6** chosen by peak silhouette.
- **Profiling:** per-cluster continuous means, dominant one-hot category per family, in-cluster approval rate, and a **leakage-safe** demographic cross-tab (cluster pipeline fit on the train sample, then applied to `X_val`).
- **Decision deferred:** whether cluster ID becomes an engineered feature is left for later (would need its own train-fit transform).


In [4]:
# 1) stratified 1M sample from train (preserve ~74.6% approval)
X_samp, _, y_samp, _ = train_test_split(X_train, y_train, train_size=1_000_000,
                                        random_state=42, stratify=y_train)
print('sample approval%% =', round(y_samp.mean(), 4), '| n =', len(X_samp))
assert abs(y_samp.mean() - 0.7465) < 0.005

# 2) preprocessing: standardize continuous, keep one-hot as-is, then PCA (90%% var)
scaler = StandardScaler().fit(X_samp[CONT_FEATURES])
Xc = scaler.transform(X_samp[CONT_FEATURES])
Xoh = X_samp.drop(columns=CONT_FEATURES).values
Xs = np.hstack([Xc, Xoh]).astype(np.float32)
pca = PCA(n_components=0.90, random_state=42).fit(Xs)
Xp = pca.transform(Xs)
print('PCA components for 90%% var:', pca.n_components_)

# 3) choose k via inertia + silhouette on a 15k subsample (50k would OOM on pairwise distances)
Xsub, _, _, _ = train_test_split(Xp, y_samp, train_size=15_000, random_state=42, stratify=y_samp)
ks = range(4, 11); inertias = []; sils = []
for k in ks:
    kmk = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3).fit(Xsub)
    inertias.append(kmk.inertia_)
    sils.append(silhouette_score(Xsub, kmk.labels_))
best_k = ks[int(np.argmax(sils))]
print('inertia by k:', dict(zip(ks, [round(i / 1e9, 2) for i in inertias])))
print('silhouette by k:', dict(zip(ks, [round(s, 3) for s in sils])), '-> chosen k =', best_k)

# 4) fit MiniBatchKMeans on the full 1M sample
km = MiniBatchKMeans(n_clusters=best_k, random_state=42, n_init=5).fit(Xp)
labels = km.labels_

# 5) profile clusters (continuous means + in-cluster approval rate)
prof = X_samp[CONT_FEATURES].copy(); prof['cluster'] = labels; prof['approved'] = y_samp
cp = prof.groupby('cluster').agg({**{c: 'mean' for c in CONT_FEATURES}, 'approved': 'mean'})
cp['size'] = prof.groupby('cluster').size()
cp = cp.rename(columns={'approved': 'approval_rate'})
# dominant one-hot category per categorical family
fam_cols = {}
for c in X_samp.columns:
    if c not in CONT_FEATURES:
        fam_cols.setdefault(c.split('_', 1)[0], []).append(c)
dom = {g: {fam: X_samp[fam_cols[fam]].columns[X_samp[fam_cols[fam]].values[labels == g].mean(axis=0).argmax()]
           for fam in fam_cols} for g in np.unique(labels)}
print(cp.round(3))
print(pd.DataFrame(dom).T)
cp.to_csv(M + 'day5_cluster_profile.csv', index=True)
print('saved day5_cluster_profile.csv')

# 6) demographic composition on VAL (leakage-safe: pipeline fit on train sample, applied to X_val)
Xc_v = scaler.transform(X_val[CONT_FEATURES].values)
Xoh_v = X_val.drop(columns=CONT_FEATURES).values
Xs_v = np.hstack([Xc_v, Xoh_v]).astype(np.float32)
Xp_v = pca.transform(Xs_v)
val_labels = km.predict(Xp_v)
vd = val_demo.copy(); vd['cluster'] = val_labels
print('Cluster x race (row-normalized):')
print(pd.crosstab(vd['cluster'], vd['derived_race'], normalize='index').round(3))
print('Cluster x sex (row-normalized):')
print(pd.crosstab(vd['cluster'], vd['derived_sex'], normalize='index').round(3))


sample approval%% = 0.7465 | n = 1000000
PCA components for 90%% var: 26
inertia by k: {4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0, 10: 0.0}
silhouette by k: {4: 0.085, 5: 0.086, 6: 0.091, 7: 0.079, 8: 0.088, 9: 0.058, 10: 0.09} -> chosen k = 6
           loan_amount  loan_to_value_ratio  property_value      income  \
cluster                                                                   
0        327409.531250            79.084000    4.619992e+05  132.442001   
1        127858.218750            61.178001    5.408110e+05  135.362000   
2        103532.492188            61.762001    4.227448e+05  118.134003   
3        983503.187500            68.629997    1.695173e+06  458.191010   
4        266670.531250            81.523003    3.562204e+05  106.596001   
5        207866.218750            78.750000    3.847788e+05  134.031006   

         debt_to_income_ratio   loan_term  total_units  \
cluster                                                  
0                   43.076000  350

/Volumes/Mitul/Projects/home-mortage-approval-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## **Step 2 - Logistic Regression baseline**
- `LogisticRegression(class_weight='balanced', solver='saga', max_iter=100)` fit on **standardized** `X_train` (scaling needed for `saga` convergence and for comparable coefficients).
- Coefficients ranked by absolute magnitude; near-zero flag = bottom 20% of |coef|.
- **VIF on continuous features only** (manual 1/(1-R^2) via `LinearRegression`). One-hot groups sum to 1 by construction, so their VIF is undefined and is deliberately excluded.


In [5]:
# scale all features for LR (fit on train only)
sc = StandardScaler().fit(X_train)
lr = LogisticRegression(class_weight='balanced', solver='saga', max_iter=100, n_jobs=-1, random_state=42)
lr.fit(sc.transform(X_train), y_train)
results['LR'] = evaluate_model(lr, sc.transform(X_val), y_val, 'LogisticRegression')

coef = pd.Series(lr.coef_[0], index=X_train.columns)
coef_df = pd.DataFrame({'feature': coef.index, 'coef': coef.values,
                        'abs_coef': coef.abs().values, 'direction': np.sign(coef.values)})
coef_df = coef_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)
coef_df['rank'] = np.arange(1, len(coef_df) + 1)
coef_df['near_zero'] = coef_df['abs_coef'] < coef_df['abs_coef'].quantile(0.20)
print(coef_df.head(15))

# VIF on continuous features only (manual: 1/(1-R^2) via LinearRegression; avoids statsmodels dep)
def vif_frame(X):
    out = []
    for c in X.columns:
        yv = X[c].values; Xv = X.drop(columns=c).values
        r2 = LinearRegression().fit(Xv, yv).score(Xv, yv)
        out.append((c, 1.0 / (1.0 - r2) if r2 < 1 else float('inf')))
    return pd.DataFrame(out, columns=['feature', 'VIF']).sort_values('VIF', ascending=False)

vif = vif_frame(X_train[CONT_FEATURES])
print('VIF (continuous features only):')
print(vif)
vif.to_csv(M + 'day5_vif_continuous.csv', index=False)


/Volumes/Mitul/Projects/home-mortage-approval-predictor/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[LogisticRegression] ROC-AUC=0.8012 PR-AUC=0.9127 P=0.8869 R=0.7225 F1=0.7963
                              feature      coef  abs_coef  direction  rank  \
0        debt_to_income_ratio_missing -0.528385  0.528385       -1.0     1   
1                       preapproval_2 -0.471524  0.471524       -1.0     2   
2                       preapproval_1  0.471524  0.471524        1.0     3   
3                      loan_purpose_1  0.419954  0.419954        1.0     4   
4                loan_to_income_ratio -0.333558  0.333558       -1.0     5   
5                      loan_purpose_4 -0.275326  0.275326       -1.0     6   
6                     loan_purpose_32 -0.191720  0.191720       -1.0     7   
7               applicant_age_Unknown  0.185937  0.185937        1.0     8   
8                 loan_to_value_ratio -0.183569  0.183569       -1.0     9   
9   tract_minority_population_percent -0.172684  0.172684       -1.0    10   
10      applicant_credit_score_type_8 -0.171429  0.171429       

## **Step 3 - Decision Tree baseline**
- Shallow, interpretable `DecisionTreeClassifier(max_depth=6, class_weight='balanced')` - depth capped for readability, not performance.
- `feature_importances_` ranked and compared to the LR ranking via **Spearman correlation**; disagreement flags likely non-linear / interaction effects the linear model can't see. Top 3 levels printed for plain-language interpretation.


In [6]:
dt = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
results['DT'] = evaluate_model(dt, X_val, y_val, 'DecisionTree')

dt_imp = pd.Series(dt.feature_importances_, index=X_train.columns).sort_values(ascending=False).reset_index()
dt_imp.columns = ['feature', 'importance']; dt_imp['rank'] = np.arange(1, len(dt_imp) + 1)
print(dt_imp.head(15))

# ranking comparison LR vs DT (Spearman)
lr_rank = coef_df.set_index('feature')['rank']
dt_rank = dt_imp.set_index('feature')['rank']
rho, p = spearmanr(lr_rank, dt_rank)
print(f"Spearman LR-rank vs DT-rank: rho={rho:.3f} p={p:.2e}")

print(export_text(dt, feature_names=list(X_train.columns), max_depth=3))


[DecisionTree] ROC-AUC=0.8051 PR-AUC=0.9045 P=0.8873 R=0.7327 F1=0.8026
                                              feature  importance  rank
0   derived_dwelling_category_Single Family (1-4 U...    0.281534     1
1                                      loan_purpose_1    0.266876     2
2                        debt_to_income_ratio_missing    0.112900     3
3                                              income    0.095526     4
4                                loan_to_income_ratio    0.044470     5
5                                 loan_to_value_ratio    0.041175     6
6                                      property_value    0.031471     7
7                                     loan_purpose_31    0.019191     8
8                                  reverse_mortgage_2    0.017952     9
9                                           loan_term    0.017375    10
10                          open_end_line_of_credit_1    0.012479    11
11                                        loan_type_3    0.00916

## **Step 4 - Random Forest baseline**
- Treated as the bridge model whose importance signal is more trustworthy than a single tree's.
- Fit on a **1.5M stratified subsample** (full 5.95M is impractical for Day-5 runtime; documented trade-off, revisitable in Day 6).
- `RandomForestClassifier(n_estimators=150, class_weight='balanced_subsample', n_jobs=-1)`; importances added to the running comparison.


In [7]:
Xrf, _, yrf, _ = train_test_split(X_train, y_train, train_size=1_500_000,
                                  random_state=42, stratify=y_train)
print('RF subsample:', Xrf.shape, '| approval%%', round(yrf.mean(), 4))
rf = RandomForestClassifier(n_estimators=150, class_weight='balanced_subsample',
                            n_jobs=-1, random_state=42)
rf.fit(Xrf, yrf)
results['RF'] = evaluate_model(rf, X_val, y_val, 'RandomForest')

rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).reset_index()
rf_imp.columns = ['feature', 'importance']; rf_imp['rank'] = np.arange(1, len(rf_imp) + 1)
print(rf_imp.head(15))


RF subsample: (1500000, 103) | approval%% 0.7465
[RandomForest] ROC-AUC=0.8732 PR-AUC=0.9417 P=0.8569 R=0.9652 F1=0.9078
                              feature  importance  rank
0                              income    0.073533     1
1                loan_to_income_ratio    0.066968     2
2                 loan_to_value_ratio    0.060492     3
3   tract_minority_population_percent    0.052007     4
4                         loan_amount    0.051746     5
5                      property_value    0.048762     6
6                    tract_population    0.044746     7
7      tract_one_to_four_family_homes    0.044688     8
8          tract_owner_occupied_units    0.044122     9
9      tract_to_msa_income_percentage    0.042970    10
10  ffiec_msa_md_median_family_income    0.041893    11
11                     loan_purpose_1    0.041227    12
12  tract_median_age_of_housing_units    0.039033    13
13       debt_to_income_ratio_missing    0.032117    14
14               debt_to_income_ratio  

## **Step 5 - Combined ranking & feature-selection recommendation**
- Builds `feature | LR_rank | DT_rank | RF_rank | composite_rank` and recommends **drop** for features in the bottom quartile of composite rank (largest rank numbers = least important across all three methods).
- **Non-linear keep:** `RF_rank` in top 25% **and** `LR_rank` in bottom 25% -> "keep (likely non-linear/interaction)".
- Ranking caveat: one-hot dummies are ranked individually; evaluate entire categorical *groups* before dropping a single level.
- **This cell only produces the recommendation - it is not applied to the data** (that's a Day-6 decision).


In [8]:
combined = pd.DataFrame({
    'LR_rank': coef_df.set_index('feature')['rank'],
    'DT_rank': dt_imp.set_index('feature')['rank'],
    'RF_rank': rf_imp.set_index('feature')['rank']}).fillna(len(X_train.columns))
combined['composite_rank'] = combined.mean(axis=1)
combined = combined.sort_values('composite_rank')
n = len(X_train.columns)
# NOTE: composite_rank is an AVERAGE RANK NUMBER (1 = most important). The least
# important features have the LARGEST rank numbers, so "drop" = top quartile of rank numbers.
q_drop = combined['composite_rank'].quantile(0.75)
combined['recommendation'] = 'keep'
combined.loc[combined['composite_rank'] >= q_drop, 'recommendation'] = 'drop (low signal across methods)'
combined.loc[(combined['RF_rank'] <= n * 0.25) & (combined['LR_rank'] > n * 0.75),
             'recommendation'] = 'keep (likely non-linear/interaction)'
print(combined.head(20))
print('Drop candidates:', list(combined[combined['recommendation'].str.startswith('drop')].index))
combined.to_csv(M + 'day5_feature_selection.csv', index=True)
print('VIF caveat: one-hot groups sum to 1 -> VIF undefined by construction; VIF table covers continuous features only.')
print('Ranking caveat: one-hot dummies are ranked individually; evaluate entire categorical groups, not single levels, before dropping.')


                                                    LR_rank  DT_rank  RF_rank  \
feature                                                                         
loan_to_income_ratio                                      5        5        2   
debt_to_income_ratio_missing                              1        3       14   
loan_purpose_1                                            4        2       12   
loan_to_value_ratio                                       9        6        3   
income                                                   14        4        1   
loan_amount                                              13       18        5   
property_value                                           23        7        6   
derived_dwelling_category_Single Family (1-4 Un...       18        1       19   
applicant_credit_score_type_8                            11       17       22   
loan_term                                                27       10       16   
co_applicant_credit_score_ty

## **Step 6 - Model comparison & plots**
- Consolidated metrics table across LR / DT / RF on `X_val`.
- Overlaid ROC and PR curves (one figure each) saved to `figures/day5/`.


In [9]:
metrics = pd.DataFrame({k: results[k] for k in results},
                            index=['roc_auc', 'pr_auc', 'precision', 'recall', 'f1']).T
print(metrics.round(4))

plt.figure(figsize=(6, 6))
for m_, r in results.items():
    fpr, tpr, _ = roc_curve(y_val, r['proba'])
    plt.plot(fpr, tpr, label=f"{m_} (AUC={r['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend()
plt.title('ROC - Day5 baselines (X_val)'); plt.savefig(FIG + 'roc_overlay.png', dpi=120); plt.close()

plt.figure(figsize=(6, 6))
for m_, r in results.items():
    prec, rec_, _ = precision_recall_curve(y_val, r['proba'])
    plt.plot(rec_, prec, label=f"{m_} (AP={r['pr_auc']:.3f})")
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend()
plt.title('PR - Day5 baselines (X_val)'); plt.savefig(FIG + 'pr_overlay.png', dpi=120); plt.close()

metrics.to_csv(M + 'day5_baseline_metrics.csv')
print('saved overlays + metrics')


    roc_auc  pr_auc  precision  recall      f1
LR   0.8012  0.9127     0.8869  0.7225  0.7963
DT   0.8051  0.9045     0.8873  0.7327  0.8026
RF   0.8732  0.9417     0.8569  0.9652  0.9078
saved overlays + metrics


## **Step 7 - Wrap-up**
- Prints the Day-5 summary and writes it to `markdown/day5_baseline_summary.md`.
- Surfaces open questions for Day 6 (does dropping low-signal features hurt the boosted models? do any clusters warrant separate modeling? engineer interactions or let boosted models find them?).


In [10]:
summary_lines = [
    "# Day 5 Baseline Models - Summary",
    "",
    f"- Best baseline on X_val by ROC-AUC: {metrics['roc_auc'].idxmax()} ({metrics['roc_auc'].max():.4f})",
    "- Metrics (X_val):",
    metrics.round(4).to_string(),
    "",
    f"- Clustering: k={best_k} via silhouette (15k subsample); preprocessing = StandardScaler(continuous)+PCA({pca.n_components_} comps, 90% var)+MiniBatchKMeans. Demographic composition on VAL (leakage-safe).",
    f"- Feature-selection recommendation counts: {combined['recommendation'].value_counts().to_dict()}",
    f"- Drop candidates (low signal all 3 methods): {list(combined[combined['recommendation'].str.startswith('drop')].index)}",
    "- Open questions for Day 6:",
    "  1. Does dropping 'drop candidate' features hurt LightGBM/CatBoost or confirm noise?",
    "  2. Do any Step-0 clusters warrant separate modeling vs just monitoring?",
    "  3. Given RF importances, engineer explicit interactions or let boosted models discover them?",
]
summary = "\n".join(summary_lines)
print(summary)
with open('/Volumes/Mitul/Projects/home-mortage-approval-predictor/markdown/day5_baseline_summary.md', 'w') as f:
    f.write(summary)
print('wrote markdown/day5_baseline_summary.md')


# Day 5 Baseline Models - Summary

- Best baseline on X_val by ROC-AUC: RF (0.8732)
- Metrics (X_val):
    roc_auc  pr_auc  precision  recall      f1
LR   0.8012  0.9127     0.8869  0.7225  0.7963
DT   0.8051  0.9045     0.8873  0.7327  0.8026
RF   0.8732  0.9417     0.8569  0.9652  0.9078

- Clustering: k=6 via silhouette (15k subsample); preprocessing = StandardScaler(continuous)+PCA(26 comps, 90% var)+MiniBatchKMeans. Demographic composition on VAL (leakage-safe).
- Feature-selection recommendation counts: {'keep': 77, 'drop (low signal across methods)': 26}
- Drop candidates (low signal all 3 methods): ['loan_type_4', 'co_applicant_credit_score_type_8', 'co_applicant_credit_score_type_6', 'applicant_credit_score_type_14', 'negative_amortization_2', 'co_applicant_credit_score_type_12', 'co_applicant_credit_score_type_Exempt', 'co_applicant_credit_score_type_15', 'prepayment_penalty_term_missing', 'balloon_payment_Exempt', 'negative_amortization_Exempt', 'interest_only_payment_Exempt

## Day 5 - Results summary
**Baselines (evaluated on `X_val`):**

| Model | ROC-AUC | PR-AUC | Precision | Recall | F1 |
|---|---|---|---|---|---|
| Logistic Regression | 0.8012 | 0.9127 | 0.8869 | 0.7225 | 0.7963 |
| Decision Tree (d=6) | 0.8051 | 0.9045 | 0.8873 | 0.7327 | 0.8026 |
| Random Forest | **0.8732** | **0.9417** | 0.8569 | **0.9652** | **0.9078** |

- **Strongest baseline: Random Forest** (ROC-AUC 0.873). Linear and single-tree baselines are ~0.80 - expected, since approval is driven by non-linear/interaction structure the linear model can't capture.
- **Clustering (k=6, PCA 26 comps):** segments range from ~28k to ~333k rows. Approval rates span **0.621 -> 0.862** vs overall 0.7465. The highest-approval clusters are larger-loan / lower-LTI profiles; the lowest (clusters 1-2) are smaller-loan, higher-denial segments. Demographic composition reported on `X_val` (row-normalized cross-tabs) - descriptive only, not a fairness test.
- **VIF (continuous only):** max ~= 6.44 (`tract_owner_occupied_units`); all < 10 -> no severe multicollinearity among continuous features.
- **Feature-selection recommendation:** 77 keep / 26 drop (low signal across all three methods). Drop candidates are mostly individual one-hot dummy levels (several `co_applicant_credit_score_type_*`, `applicant_credit_score_type_*`, `negative_amortization_*`, `Exempt` levels) plus a few `*_missing` flags - **no core continuous feature (income, loan_amount, LTV, LTI, tract_*) is recommended for dropping.** No feature hit the "non-linear-only" flag at the 25% threshold.
- **Next:** Day 6 takes the recommendation under consideration and moves to LightGBM/CatBoost (with `X_val` for tuning/early-stopping and `X_test` reserved for final + Day-9 fairness).
